# Nearest Neighbors
In this section we'll develop the *nearest neighbor* method of classification. Just focus on the ideas for now and don't worry if some of the code is mysterious. Later in the chapter we'll see how to organize our ideas into code that performs the classification.

In [1]:
import polars as pl
import numpy as np
from datascience import make_array
import hvplot.polars
import holoviews as hv
path_data = '../../../assets/data/'

# POLARS_CHANGE: hvplot colors each point by its Color column; this maps the Data 8 color names to Data 8 blue and gold.
color_map = {'darkblue': '#002fa7', 'gold': '#706700'}

In [2]:
def standard_units(x):
    """POLARS_CHANGE: x is a column (a Polars Series), so this uses Series methods. std(ddof=0) matches np.std."""
    return (x - x.mean())/x.std(ddof=0)

In [3]:
def distance(point1, point2):
    """The distance between two arrays of numbers."""
    return np.sqrt(np.sum((point1 - point2)**2))

def all_distances(training, point):
    """The distance between p (an array of numbers) and the numbers in row i of attribute_table."""
    attributes = training.drop('Class')
    def distance_from_point(row):
        return distance(point, np.array(row))
    # POLARS_CHANGE: iter_rows passes each row as a tuple; collect the distances in an array.
    distances = make_array()
    for row in attributes.iter_rows():
        distances = np.append(distances, distance_from_point(row))
    return distances

def table_with_distances(training, point):
    """A copy of the training table with the distance from each row to array p."""
    return training.with_columns(pl.Series('Distance', all_distances(training, point)))

def closest(training, point, k):
    """A table containing the k closest rows in the training table to array p."""
    with_dists = table_with_distances(training, point)
    sorted_by_distance = with_dists.sort('Distance')
    topk = sorted_by_distance.head(k)
    return topk

def majority(topkclasses):
    """1 if the majority of the "Class" column is 1s, and 0 otherwise."""
    ones = topkclasses.filter(pl.col('Class') == 1).height
    zeros = topkclasses.filter(pl.col('Class') == 0).height
    if ones > zeros:
        return 1
    else:
        return 0

def classify(training, p, k):
    """Classify an example with attributes p using k-nearest neighbor classification with the given training table."""
    closestk = closest(training, p, k)
    topkclasses = closestk.select('Class')
    return majority(topkclasses)

## Chronic kidney disease

Let's work through an example.  We're going to work with a data set that was collected to help doctors diagnose chronic kidney disease (CKD).  Each row in the data set represents a single patient who was treated in the past and whose diagnosis is known.  For each patient, we have a bunch of measurements from a blood test.  We'd like to find which measurements are most useful for diagnosing CKD, and develop a way to classify future patients as "has CKD" or "doesn't have CKD" based on their blood test results.

In [4]:
ckd = pl.read_csv(path_data + 'ckd.csv').rename({'Blood Glucose Random': 'Glucose'})
# POLARS_CHANGE: two White Blood Cell Count entries start with a tab, so Polars reads that column as text.
# Strip the tab and convert the column to integers.
ckd = ckd.with_columns(pl.col('White Blood Cell Count').str.strip_chars().cast(pl.Int64))
ckd

Age,Blood Pressure,Specific Gravity,Albumin,Sugar,Red Blood Cells,Pus Cell,Pus Cell clumps,Bacteria,Glucose,Blood Urea,Serum Creatinine,Sodium,Potassium,Hemoglobin,Packed Cell Volume,White Blood Cell Count,Red Blood Cell Count,Hypertension,Diabetes Mellitus,Coronary Artery Disease,Appetite,Pedal Edema,Anemia,Class
i64,i64,f64,i64,i64,str,str,str,str,i64,i64,f64,i64,f64,f64,i64,i64,f64,str,str,str,str,str,str,i64
48,70,1.005,4,0,"""normal""","""abnormal""","""present""","""notpresent""",117,56,3.8,111,2.5,11.2,32,6700,3.9,"""yes""","""no""","""no""","""poor""","""yes""","""yes""",1
53,90,1.02,2,0,"""abnormal""","""abnormal""","""present""","""notpresent""",70,107,7.2,114,3.7,9.5,29,12100,3.7,"""yes""","""yes""","""no""","""poor""","""no""","""yes""",1
63,70,1.01,3,0,"""abnormal""","""abnormal""","""present""","""notpresent""",380,60,2.7,131,4.2,10.8,32,4500,3.8,"""yes""","""yes""","""no""","""poor""","""yes""","""no""",1
68,80,1.01,3,2,"""normal""","""abnormal""","""present""","""present""",157,90,4.1,130,6.4,5.6,16,11000,2.6,"""yes""","""yes""","""yes""","""poor""","""yes""","""no""",1
61,80,1.015,2,0,"""abnormal""","""abnormal""","""notpresent""","""notpresent""",173,148,3.9,135,5.2,7.7,24,9200,3.2,"""yes""","""yes""","""yes""","""poor""","""yes""","""yes""",1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
55,80,1.02,0,0,"""normal""","""normal""","""notpresent""","""notpresent""",140,49,0.5,150,4.9,15.7,47,6700,4.9,"""no""","""no""","""no""","""good""","""no""","""no""",0
42,70,1.025,0,0,"""normal""","""normal""","""notpresent""","""notpresent""",75,31,1.2,141,3.5,16.5,54,7800,6.2,"""no""","""no""","""no""","""good""","""no""","""no""",0
12,80,1.02,0,0,"""normal""","""normal""","""notpresent""","""notpresent""",100,26,0.6,137,4.4,15.8,49,6600,5.4,"""no""","""no""","""no""","""good""","""no""","""no""",0


Some of the variables are categorical (words like "abnormal"), and some quantitative. The quantitative variables all have different scales. We're going to want to make comparisons and estimate distances, often by eye, so let's select just a few of the variables and work in standard units. Then we won't have to worry about the scale of each of the different variables.

In [5]:
# Each standardized Series keeps the name of the column it came from.
ckd = pl.DataFrame([
    standard_units(ckd.get_column('Hemoglobin')),
    standard_units(ckd.get_column('Glucose')),
    standard_units(ckd.get_column('White Blood Cell Count')),
    ckd.get_column('Class'),
])

In [6]:
ckd

Hemoglobin,Glucose,White Blood Cell Count,Class
f64,f64,f64,i64
-0.865744,-0.221549,-0.569768,1
-1.457446,-0.947597,1.162684,1
-1.004968,3.841231,-1.275582,1
-2.814879,0.396364,0.809777,1
-2.083954,0.643529,0.232293,1
…,…,…,…
0.700526,0.133751,-0.569768,0
0.978974,-0.870358,-0.216861,0
0.735332,-0.484162,-0.60185,0


Let's look at two columns in particular: the hemoglobin level (in the patient's blood), and the blood glucose level (at a random time in the day; without fasting specially for the blood test). 

We'll draw a scatter plot to visualize the relation between the two variables. Blue dots are patients with CKD; gold dots are patients without CKD.  What kind of medical test results seem to indicate CKD?

In [7]:
color_table = pl.DataFrame([
    pl.Series('Class', make_array(1, 0)),
    pl.Series('Color', make_array('darkblue', 'gold')),
])
# POLARS_CHANGE: Data 8 join sorts the rows by the joined column. Polars keeps the order of the left table, so sort by Class.
ckd = ckd.join(color_table, on='Class', maintain_order='left').sort('Class', maintain_order=True)

In [8]:
ckd.hvplot.scatter(x='Hemoglobin', y='Glucose', color='Color', cmap=color_map)

Scatterplot with 'Hemoglobin' on the x-axis and 'Glucose' on the y-axis. Data points are shown in dark blue or in gold. The dark blue data points exist all over the graph, but not where the gold data points do, from about x=0 to x=1.5 and y values between -1 and just above 0.

Suppose Alice is a new patient who is not in the data set.  If I tell you Alice's hemoglobin level and blood glucose level, could you predict whether she has CKD?  It sure looks like it!  You can see a very clear pattern here: points in the lower-right tend to represent people who don't have CKD, and the rest tend to be folks with CKD.  To a human, the pattern is obvious.  But how can we program a computer to automatically detect patterns such as this one?

## A Nearest Neighbor Classifier

There are lots of kinds of patterns one might look for, and lots of algorithms for classification.  But I'm going to tell you about one that turns out to be surprisingly effective.  It is called *nearest neighbor classification*.  Here's the idea.  If we have Alice's hemoglobin and glucose numbers, we can put her somewhere on this scatterplot; the hemoglobin is her x-coordinate, and the glucose is her y-coordinate.  Now, to predict whether she has CKD or not, we find the nearest point in the scatterplot and check whether it is blue or gold; we predict that Alice should receive the same diagnosis as that patient.

In other words, to classify Alice as CKD or not, we find the patient in the training set who is "nearest" to Alice, and then use that patient's diagnosis as our prediction for Alice.  The intuition is that if two points are near each other in the scatterplot, then the corresponding measurements are pretty similar, so we might expect them to receive the same diagnosis (more likely than not).  We don't know Alice's diagnosis, but we do know the diagnosis of all the patients in the training set, so we find the patient in the training set who is most similar to Alice, and use that patient's diagnosis to predict Alice's diagnosis.

In the graph below, the red dot represents Alice. It is joined with a black line to the point that is nearest to it – its *nearest neighbor* in the training set. The figure is drawn by a function called `show_closest`. It takes an array that represents the $x$ and $y$ coordinates of Alice's point. Vary those to see how the closest point changes! Note especially when the closest point is blue and when it is gold.

In [9]:
def show_closest(point):
    """point = array([x,y]) 
    gives the coordinates of a new point
    shown in red"""
    
    HemoGl = ckd.drop('White Blood Cell Count', 'Color')
    t = closest(HemoGl, point, 1)
    x_closest = t.get_column('Hemoglobin').item(0)
    y_closest = t.get_column('Glucose').item(0)
    # POLARS_CHANGE: the extra point and line are HoloViews elements laid over the hvplot scatter with *.
    return (
        ckd.hvplot.scatter(x='Hemoglobin', y='Glucose', color='Color', cmap=color_map)
        * hv.Curve((make_array(point.item(0), x_closest), make_array(point.item(1), y_closest))).opts(color='black', line_width=2)
        * hv.Scatter(([point.item(0)], [point.item(1)])).opts(color='red', size=8)
    )

In [10]:
# In this example, Alice's Hemoglobin attribute is 0 and her Glucose is 1.5.
alice = make_array(0, 1.5)
show_closest(alice)

The same scatterplot as before is reproduced here with an additional data point in red at about (0, 1.5). The red data point has a black line drawn to the nearest data point in dark blue. The lins isn't that long. The previous description is: Scatterplot with 'Hemoglobin' on the x-axis and 'Glucose' on the y-axis. Data points are shown in dark blue or in gold. The dark blue data points exist all over the graph, but not where the gold data points do, from about x=0 to x=1.5 and y values between -1 and just above 0.

Thus our *nearest neighbor classifier* works like this:
- Find the point in the training set that is nearest to the new point.
- If that nearest point is a "CKD" point, classify the new point as "CKD". If the nearest point is a "not CKD" point, classify the new point as "not CKD".

The scatterplot suggests that this nearest neighbor classifier should be pretty accurate.  Points in the lower-right will tend to receive a "no CKD" diagnosis, as their nearest neighbor will be a gold point.  The rest of the points will tend to receive a "CKD" diagnosis, as their nearest neighbor will be a blue point.  So the nearest neighbor strategy seems to capture our intuition pretty well, for this example.

## Decision boundary

Sometimes a helpful way to visualize a classifier is to map out the kinds of attributes where the classifier would predict 'CKD', and the kinds where it would predict 'not CKD'.  We end up with some boundary between the two, where points on one side of the boundary will be classified 'CKD' and points on the other side will be classified 'not CKD'.  This boundary is called the *decision boundary*.  Each different classifier will have a different decision boundary; the decision boundary is just a way to visualize what criteria the classifier is using to classify points.

For example, suppose the coordinates of Alice's point are (0, 1.5). Notice that the nearest neighbor is blue. Now try reducing the height (the $y$-coordinate) of the point. You'll see that at around $y = 0.95$ the nearest neighbor turns from blue to gold.

In [11]:
alice = make_array(0, 0.97)
show_closest(alice)

The same scatterplot as before is reproduced here with an additional data point in red at about (0, 1). The red data point has a black line drawn to the nearest data point in dark blue. The lins is a bit longer than the previous black line. The previous description is: Scatterplot with 'Hemoglobin' on the x-axis and 'Glucose' on the y-axis. Data points are shown in dark blue or in gold. The dark blue data points exist all over the graph, but not where the gold data points do, from about x=0 to x=1.5 and y values between -1 and just above 0.

Here are hundreds of new unclassified points, all in red.

In [12]:
x_array = make_array()
y_array = make_array()
for x in np.arange(-2, 2.1, 0.1):
    for y in np.arange(-2, 2.1, 0.1):
        x_array = np.append(x_array, x)
        y_array = np.append(y_array, y)
        
test_grid = pl.DataFrame([
    pl.Series('Hemoglobin', x_array),
    pl.Series('Glucose', y_array),
])

In [13]:
(
    test_grid.hvplot.scatter(x='Hemoglobin', y='Glucose', color='red', alpha=0.4, size=30)
    * ckd.hvplot.scatter(x='Hemoglobin', y='Glucose', color='Color', cmap=color_map, line_color='black')
).opts(xlim=(-2, 2), ylim=(-2, 2))

A scatterplot with 'hemoglobin' on the x-axis and 'Glucose' on the y-axis. Red points form a grid over the graph. On top of the red points are dark blue data points, primarily on the left hand side of the graph. There are also gold data points, only on the right hand side of the graph. One dark blue data point has a high x value on the right hand side of the graph, but is above the gold data points with a higher y value.

Each of the red points has a nearest neighbor in the training set (the same blue and gold points as before). For some red points you can easily tell whether the nearest neighbor is blue or gold. For others, it's a little more tricky to make the decision by eye. Those are the points near the decision boundary.

But the computer can easily determine the nearest neighbor of each point. So let's get it to apply our nearest neighbor classifier to each of the red points: 

For each red point, it must find the closest point in the training set; it must then change the color of the red point to become the color of the nearest neighbor. 

The resulting graph shows which points will get classified as 'CKD' (all the blue ones), and which as 'not CKD' (all the gold ones).

In [14]:
def classify_grid(training, test, k):
    c = make_array()
    for i in np.arange(test.height):
        # Run the classifier on the ith patient in the test set
        c = np.append(c, classify(training, np.array(test.row(i)), k))
    return c

In [15]:
c = classify_grid(ckd.drop('White Blood Cell Count', 'Color'), test_grid, 1)

In [16]:
# POLARS_CHANGE: c holds floats (it started as make_array()); cast to integers so it joins with the integer Class in color_table.
test_grid = test_grid.with_columns(pl.Series('Class', c).cast(pl.Int64)).join(color_table, on='Class')
(
    test_grid.hvplot.scatter(x='Hemoglobin', y='Glucose', color='Color', cmap=color_map, alpha=0.4, size=30)
    * ckd.hvplot.scatter(x='Hemoglobin', y='Glucose', color='Color', cmap=color_map, line_color='black')
).opts(xlim=(-2, 2), ylim=(-2, 2))

Scatterplot with 'Hemoglobin' on the x-axis and 'Glucose' on the y-axis. The same dark blue and gold data points are drawn. More transparent data points are drawn in a grid behind these, either colored blue or gold, depending on if they're closest to a dark blue data point or a full gold data point. The left hand side of the graph is mostly blue, along with the top third. The rest of the graph is the lighter gold.

The decision boundary is where the classifier switches from turning the red points blue to turning them gold.

## k-Nearest Neighbors

However, the separation between the two classes won't always be quite so clean.  For instance, suppose that instead of hemoglobin levels we were to look at white blood cell count.  Look at what happens:

In [17]:
ckd.hvplot.scatter(x='White Blood Cell Count', y='Glucose', color='Color', cmap=color_map)

A scatterplot with 'White Blood Cell Count' on the x-axis and 'Glucose' on the y-axis. There are gold data points in the bottom left corner of the graph, and dark blue data points all over. The gold and dark blue data points do overlap in this graph.

As you can see, non-CKD individuals are all clustered in the lower-left.  Most of the patients with CKD are above or to the right of that cluster... but not all.  There are some patients with CKD who are in the lower left of the above figure (as indicated by the handful of blue dots scattered among the gold cluster).  What this means is that you can't tell for certain whether someone has CKD from just these two blood test measurements.

If we are given Alice's glucose level and white blood cell count, can we predict whether she has CKD?  Yes, we can make a prediction, but we shouldn't expect it to be 100% accurate.  Intuitively, it seems like there's a natural strategy for predicting: plot where Alice lands in the scatter plot; if she is in the lower-left, predict that she doesn't have CKD, otherwise predict she has CKD.  

This isn't perfect -- our predictions will sometimes be wrong.  (Take a minute and think it through: for which patients will it make a mistake?)  As the scatterplot above indicates, sometimes people with CKD have glucose and white blood cell levels that look identical to those of someone without CKD, so any classifier is inevitably going to make the wrong prediction for them.

Can we automate this on a computer?  Well, the nearest neighbor classifier would be a reasonable choice here too.  Take a minute and think it through: how will its predictions compare to those from the intuitive strategy above?  When will they differ?

Its predictions will be pretty similar to our intuitive strategy, but occasionally it will make a different prediction.  In particular, if Alice's blood test results happen to put her right near one of the blue dots in the lower-left, the intuitive strategy would predict 'not CKD', whereas the nearest neighbor classifier will predict 'CKD'.

There is a simple generalization of the nearest neighbor classifier that fixes this anomaly.  It is called the *k-nearest neighbor classifier*.  To predict Alice's diagnosis, rather than looking at just the one neighbor closest to her, we can look at the 3 points that are closest to her, and use the diagnosis for each of those 3 points to predict Alice's diagnosis.  In particular, we'll use the majority value among those 3 diagnoses as our prediction for Alice's diagnosis.  Of course, there's nothing special about the number 3: we could use 4, or 5, or more.  (It's often convenient to pick an odd number, so that we don't have to deal with ties.)  In general, we pick a number $k$, and our predicted diagnosis for Alice is based on the $k$ patients in the training set who are closest to Alice.  Intuitively, these are the $k$ patients whose blood test results were most similar to Alice, so it seems reasonable to use their diagnoses to predict Alice's diagnosis.

The $k$-nearest neighbor classifier will now behave just like our intuitive strategy above.